# Ensemble Model (Voting Classifier)

Xây dựng mô hình Ensemble kết hợp 3 mô hình tốt nhất: **XGBoost**, **LightGBM**, và **Random Forest** sử dụng phương pháp **Soft Voting** để tăng độ ổn định của dự đoán.

In [ ]:
import sys
import time
import os
import pickle
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import lightgbm as lgb
import joblib

# Thêm thư mục gốc dự án vào sys.path
sys.path.insert(0, str(Path.cwd()))

from src.model_training import (
    load_splits,
    evaluate_model,
    plot_confusion_matrix,
    compare_models,
)

In [ ]:
# --- STEP 1: Load dữ liệu đã chia sẵn ---
print("=" * 70)
print("STEP 1: Loading pre-split data")
print("=" * 70)

X_train, X_test, y_train, y_test = load_splits()

print(f"Feature columns: {X_train.shape[1]}")
print(f"Train size: {len(X_train):,}  |  Test size: {len(X_test):,}")
print(f"Classes: {sorted(y_train.unique())}")

In [ ]:
# --- STEP 2: Preprocessing (Label Encoding cho XGBoost) ---
print("\n" + "=" * 70)
print("STEP 2: Preprocessing & Base Models Setup")
print("=" * 70)

# VotingClassifier yêu cầu các class numbers chung cho tất cả các mô hình bên trong.
# Vì XGBoost cần nhãn số, ta sẽ transform toàn bộ y sang dạng số.
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

print(f"Label encoding applied — {len(le.classes_)} classes")

# Khởi tạo 3 base models (dùng params baseline tốt nhất)
xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=len(le.classes_),
    n_estimators=300,
    max_depth=7,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
    eval_metric="mlogloss",
)

lgb_model = lgb.LGBMClassifier(
    num_leaves=31,
    learning_rate=0.05,
    n_estimators=300, # Tăng lên 300 cho đồng bộ
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
    verbose=0,
)

print("Base models initialized: XGBoost, LightGBM, Random Forest.")

In [ ]:
# --- STEP 3: Train Ensemble Model ---
print("\n" + "=" * 70)
print("STEP 3: Training Voting Classifier (Soft Voting)")
print("=" * 70)

# Khởi tạo VotingClassifier
# Soft voting tính trung bình xác suất (probabilities) dự đoán từ các base models
ensemble_model = VotingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('lgb', lgb_model),
        ('rf', rf_model)
    ],
    voting='soft',
    n_jobs=-1  # Train các models song song
)

t0 = time.time()
print("Training ensemble model (this will take a few minutes)...")
ensemble_model.fit(X_train, y_train_enc)
train_time = time.time() - t0

print(f"\nTraining completed in {train_time:.2f} seconds")

In [ ]:
# --- STEP 4: Evaluate ---
print("\n" + "=" * 70)
print("STEP 4: Model Evaluation")
print("=" * 70)

y_pred_enc = ensemble_model.predict(X_test)
y_pred_proba = ensemble_model.predict_proba(X_test)

# Giải mã nhãn số về chuỗi gốc
y_pred = le.inverse_transform(y_pred_enc)

ens_results = evaluate_model(
    y_true=y_test,
    y_pred=y_pred,
    model_name="Ensemble (XGB+LGBM+RF)",
    y_pred_proba=y_pred_proba,
    labels=le.classes_.tolist(),
)

In [ ]:
# --- STEP 5: Confusion Matrix ---
print("\n" + "=" * 70)
print("STEP 5: Confusion Matrix")
print("=" * 70)

plot_confusion_matrix(
    y_true=y_test,
    y_pred=y_pred,
    labels=le.classes_.tolist(),
    model_name="Voting Ensemble",
    normalize=True,
    figsize=(12, 10),
)

In [ ]:
# --- STEP 6: Summary & Save ---
print("\n" + "=" * 70)
print("STEP 6: Summary & Model Saving")
print("=" * 70)

comparison_df = compare_models([ens_results])
print("\nPerformance Metrics:")
print(comparison_df.to_string(index=False))

print(f"\nTraining time  : {train_time:.2f}s")
print(f"Test set size  : {len(y_test):,}")
print(f"Base models    : 3 (XGB, LGBM, RF)")

# Lưu mô hình (Tùy chọn)
# Chú ý: Ensemble model (với XGBoost) trên dataset lớn sẽ có dung lượng vài trăm MB
# Ta lưu best single model (XGBoost) như model chính thức thay vì Ensemble.

models_dir = Path.cwd().parent / "models"
models_dir.mkdir(exist_ok=True)

# Lấy XGBoost model đã train từ VotingClassifier
trained_xgb = ensemble_model.named_estimators_['xgb']

best_model_path = models_dir / "xgboost_best_model.joblib"
le_path = models_dir / "label_encoder.joblib"

joblib.dump(trained_xgb, best_model_path)
joblib.dump(le, le_path)

print(f"\n✓ Saved best single model (XGBoost) to {best_model_path}")
print(f"✓ Saved LabelEncoder to {le_path}")